In [1]:
# Condensed Motor Learning Analysis Pipeline
import os
os.environ['OMP_NUM_THREADS'] = '1'

from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union, Any
import json
import pickle
import warnings
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, gaussian_kde, ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Statistical packages
import pingouin as pg
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

@dataclass
class Config:
    """Configuration for analysis parameters."""
    base_output_dir: Path = Path('motor_learning_output')
    min_complete_strides: int = 20
    motor_noise_strides: int = 20
    motor_noise_threshold: float = 0.3
    success_rate_threshold: float = 0.68
    figure_dpi: int = 300
    alpha_level: float = 0.05
    
    trial_type_mapping: Dict[str, str] = field(default_factory=lambda: {
        'primer': 'vis1', 'trial': 'invis', 'vis': 'vis2', 'pref': 'pref'
    })
    
    def __post_init__(self):
        """Create directory structure."""
        self.figures_dir = self.base_output_dir / 'figures'
        self.reports_dir = self.base_output_dir / 'reports'
        self.exports_dir = self.base_output_dir / 'exports'
        self.processed_data_dir = self.base_output_dir / 'processed_data'
        
        for directory in [self.figures_dir, self.reports_dir, self.exports_dir, self.processed_data_dir]:
            directory.mkdir(parents=True, exist_ok=True)
        
        self.processed_data_file = self.processed_data_dir / 'processed_data.pkl'

class DataProcessor:
    """Handles all data loading and processing."""
    
    def __init__(self, config: Config):
        self.config = config
        
    def load_file(self, file_path: Path) -> Optional[pd.DataFrame]:
        """Load and validate a single data file."""
        try:
            df = pd.read_csv(file_path, sep='\t')
            if 'Stride Number' in df.columns:
                df['Stride Number'] = pd.to_numeric(df['Stride Number'], errors='coerce')
                df = df.dropna(subset=['Stride Number']).drop_duplicates(subset=['Stride Number']).sort_values('Stride Number')
            return df if not df.empty else None
        except Exception:
            return None

    def process_trial_files(self, subject_dir: Path, trial_prefix: str) -> Optional[pd.DataFrame]:
        """Find and combine trial files for a given trial type."""
        all_files = sorted(subject_dir.glob(f"{trial_prefix}*.txt"))
        if not all_files:
            return None
        
        if trial_prefix == 'pref':
            largest_file = max(all_files, key=lambda f: f.stat().st_size)
            return self.load_file(largest_file)
        
        if len(all_files) == 1:
            return self.load_file(all_files[0])
        
        # Multiple files - combine them
        dfs = [self.load_file(f) for f in all_files if self.load_file(f) is not None]
        if not dfs:
            return None
        
        combined = pd.concat(dfs, ignore_index=True)
        if 'Stride Number' in combined.columns:
            combined = combined.sort_values('Stride Number').drop_duplicates('Stride Number')
        return combined

    def detect_anomalies(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        """Detect and flag anomalies in stride data."""
        if df is None or df.empty:
            return df, {}
        
        df = df.copy()
        df['Anomalous'] = False
        anomalies = {}
        
        # Time-based anomalies
        time_col = next((col for col in ['Time', 'Timestamp', 'Time (s)'] if col in df.columns), None)
        if time_col:
            df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
            time_diff = df[time_col].diff()
            jump_mask = time_diff > time_diff.quantile(0.99) * 5
            
            for idx in df.index[jump_mask.fillna(False)]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('time_jump')
        
        # Sum of gains and steps anomalies
        if 'Sum of gains and steps' in df.columns:
            high_mask = df['Sum of gains and steps'] > 4
            zero_mask = df['Sum of gains and steps'] == 0
            
            for idx in df.index[high_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_high')
            
            for idx in df.index[zero_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_zero')
        
        return df, anomalies

class MetricsCalculator:
    """Calculates all metrics for motor learning analysis."""
    
    def __init__(self, config: Config):
        self.config = config
    
    def calculate_period_metrics(self, period_data: pd.DataFrame, trial_type: str, condition: str) -> Dict:
        """Calculate all metrics for a specific period."""
        metrics = {}
        
        # Success rate
        metrics[f'{trial_type}_sr_{condition}_const'] = period_data['Success'].mean()
        
        # Stride metrics
        if 'Sum of gains and steps' in period_data.columns:
            sogs = period_data['Sum of gains and steps']
            metrics[f'{trial_type}_sd_{condition}_const'] = sogs.std()
            metrics[f'{trial_type}_msl_{condition}_const'] = sogs.mean()
            
            # Learning metric calculation
            if 'Constant' in period_data.columns:
                const_value = period_data['Constant'].iloc[0] if not period_data['Constant'].empty else None
                avg_stride = sogs.mean()
                preferred_stride = 2.0
                
                if const_value is not None and not pd.isna(const_value):
                    denominator = const_value - preferred_stride
                    if abs(denominator) > 1e-6:
                        learning_value = (avg_stride - preferred_stride) / denominator
                        metrics[f'{trial_type}_learning_{condition}_const'] = learning_value
                    else:
                        metrics[f'{trial_type}_learning_{condition}_const'] = np.nan
                else:
                    metrics[f'{trial_type}_learning_{condition}_const'] = np.nan
                
                # Error calculation
                if 'Constant' in period_data.columns:
                    metrics[f'{trial_type}_error_{condition}_const'] = (sogs - period_data['Constant']).mean()
        
        # Asymmetry
        if all(col in period_data.columns for col in ['Right step length', 'Left step length']):
            asymmetry = self._calculate_asymmetry(period_data['Right step length'], period_data['Left step length'])
            if asymmetry is not None:
                metrics[f'{trial_type}_asymmetry_{condition}_const'] = asymmetry
        
        # Strides between successes
        strides_between = self._calculate_strides_between_successes(period_data)
        if strides_between is not None:
            metrics[f'{trial_type}_strides_between_success_{condition}_const'] = strides_between
        
        return metrics
    
    def calculate_preference_metrics(self, pref_df: pd.DataFrame) -> Dict:
        """Calculate metrics from preference trial."""
        metrics = {'mot_noise': None, 'pref_asymmetry': None}
        
        if pref_df is None or pref_df.empty:
            return metrics
        
        if not all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
            return metrics
        
        # Get non-zero steps
        right_steps = pref_df['Right step length']
        left_steps = pref_df['Left step length']
        
        right_clean = right_steps[(right_steps != 0) & (right_steps.notna())]
        left_clean = left_steps[(left_steps != 0) & (left_steps.notna())]
        
        min_required = self.config.motor_noise_strides
        if len(right_clean) < min_required or len(left_clean) < min_required:
            return metrics
        
        # Calculate motor noise
        final_right = right_clean.iloc[-1]
        final_left = left_clean.iloc[-1]
        
        if final_right <= 0 or final_left <= 0:
            return metrics
        
        # Normalize steps
        norm_right = right_clean / final_right
        norm_left = left_clean / final_left
        
        min_length = min(len(norm_right), len(norm_left))
        if min_length < min_required:
            return metrics
        
        sum_steps = norm_right.iloc[:min_length] + norm_left.iloc[:min_length]
        
        # Motor noise from last N points
        if len(sum_steps) >= self.config.motor_noise_strides:
            noise = sum_steps.tail(self.config.motor_noise_strides).std()
            if not pd.isna(noise) and noise > 0:
                metrics['mot_noise'] = noise
        
        # Calculate asymmetry
        if len(right_clean) >= self.config.motor_noise_strides and len(left_clean) >= self.config.motor_noise_strides:
            last_n_right = right_clean.tail(self.config.motor_noise_strides) / final_right
            last_n_left = left_clean.tail(self.config.motor_noise_strides) / final_left
            
            asymmetry = self._calculate_asymmetry(last_n_right.values, last_n_left.values)
            if asymmetry is not None:
                metrics['pref_asymmetry'] = asymmetry
        
        return metrics
    
    def calculate_retention_metrics(self, trial_data: Dict, trial_type: str) -> Dict:
        """Calculate retention metrics for invisible trial type."""
        metrics = {}
        if trial_type != 'invis':
            return metrics
        
        df = trial_data['data']
        required_cols = ['Target size', 'Constant', 'Sum of gains and steps', 'Stride Number']
        if not all(col in df.columns for col in required_cols):
            return metrics
        
        try:
            # Find maximum target size
            max_target_size = df['Target size'].max()
            target_tolerance = 0.01
            
            # Find ALL periods where target is at max size AND const = 2
            success_clamp_mask = (
                (df['Target size'] >= max_target_size - target_tolerance) & 
                (np.abs(df['Constant'] - 2.0) < target_tolerance)
            )
            
            success_clamp_data = df[success_clamp_mask].copy()
            
            if success_clamp_data.empty:
                metrics['invis_retention_clamp1_const'] = np.nan
                metrics['invis_retention_clamp2_const'] = np.nan
                return metrics
            
            # Sort by stride number and identify separate clamp periods
            success_clamp_data = success_clamp_data.sort_values('Stride Number').reset_index(drop=True)
            
            # Find breaks in stride sequence to identify separate periods
            stride_diffs = success_clamp_data['Stride Number'].diff()
            period_breaks = stride_diffs > 1
            period_breaks.iloc[0] = True
            
            # Assign period IDs
            success_clamp_data['period_id'] = period_breaks.cumsum()
            
            # Get all unique periods and filter by minimum length
            period_info = []
            for period_id in success_clamp_data['period_id'].unique():
                period_data = success_clamp_data[success_clamp_data['period_id'] == period_id]
                period_info.append({
                    'period_id': period_id,
                    'start_stride': period_data['Stride Number'].min(),
                    'end_stride': period_data['Stride Number'].max(),
                    'n_strides': len(period_data),
                    'data': period_data
                })
            
            # Sort periods by start stride to get chronological order
            period_info.sort(key=lambda x: x['start_stride'])
            
            # Filter periods with at least 20 strides
            valid_periods = [p for p in period_info if p['n_strides'] >= 20]
            
            if len(valid_periods) < 2:
                metrics['invis_retention_clamp1_const'] = np.nan
                metrics['invis_retention_clamp2_const'] = np.nan
                return metrics
            
            # Take the 2nd and 3rd periods as clamp1 and clamp2
            clamp1_period = valid_periods[1] if len(valid_periods) >= 2 else valid_periods[0]
            clamp2_period = valid_periods[2] if len(valid_periods) >= 3 else None
            
            preferred_stride = 1.0
            
            def get_const_before_clamp(clamp_start_stride):
                """Get the constant value from the stride immediately before the clamp period starts"""
                pre_clamp_stride = clamp_start_stride - 1
                pre_clamp_data = df[df['Stride Number'] == pre_clamp_stride]
                
                if not pre_clamp_data.empty:
                    return pre_clamp_data['Constant'].iloc[0]
                else:
                    earlier_data = df[df['Stride Number'] < clamp_start_stride]
                    if not earlier_data.empty:
                        closest_earlier = earlier_data.loc[earlier_data['Stride Number'].idxmax()]
                        return closest_earlier['Constant']
                    else:
                        return 2.0
            
            # Calculate retention for clamp1
            if clamp1_period is not None:
                clamp1_data = clamp1_period['data']
                last_20_clamp1 = clamp1_data.tail(20)
                avg_stride_clamp1 = last_20_clamp1['Sum of gains and steps'].mean()
                
                const_value_clamp1 = get_const_before_clamp(clamp1_period['start_stride'])
                
                denominator = const_value_clamp1 - preferred_stride
                if abs(denominator) > 1e-6:
                    retention_clamp1 = (avg_stride_clamp1 - preferred_stride) / denominator
                    metrics['invis_retention_clamp1_const'] = retention_clamp1
                else:
                    metrics['invis_retention_clamp1_const'] = np.nan
            else:
                metrics['invis_retention_clamp1_const'] = np.nan
            
            # Calculate retention for clamp2
            if clamp2_period is not None:
                clamp2_data = clamp2_period['data']
                last_20_clamp2 = clamp2_data.tail(20)
                avg_stride_clamp2 = last_20_clamp2['Sum of gains and steps'].mean()
                
                const_value_clamp2 = get_const_before_clamp(clamp2_period['start_stride'])
                
                denominator = const_value_clamp2 - preferred_stride
                if abs(denominator) > 1e-6:
                    retention_clamp2 = (avg_stride_clamp2 - preferred_stride) / denominator
                    metrics['invis_retention_clamp2_const'] = retention_clamp2
                else:
                    metrics['invis_retention_clamp2_const'] = np.nan
            else:
                metrics['invis_retention_clamp2_const'] = np.nan
            
            # Calculate overall retention as average of clamp1 and clamp2
            clamp1_ret = metrics.get('invis_retention_clamp1_const')
            clamp2_ret = metrics.get('invis_retention_clamp2_const')
            
            if (not pd.isna(clamp1_ret) and not pd.isna(clamp2_ret)):
                overall_retention = (clamp1_ret + clamp2_ret) / 2
                metrics['invis_retention_overall_const'] = overall_retention
            elif not pd.isna(clamp1_ret):
                metrics['invis_retention_overall_const'] = clamp1_ret
            else:
                metrics['invis_retention_overall_const'] = np.nan
                
        except Exception as e:
            metrics['invis_retention_clamp1_const'] = np.nan
            metrics['invis_retention_clamp2_const'] = np.nan
            metrics['invis_retention_overall_const'] = np.nan
        
        return metrics
    
    def _calculate_asymmetry(self, right_values, left_values) -> Optional[float]:
        """Calculate step length asymmetry."""
        denominator = right_values + left_values
        valid_mask = denominator != 0
        
        if not valid_mask.any():
            return None
        
        asymmetry_vals = np.abs((right_values - left_values) / denominator)[valid_mask]
        return np.mean(asymmetry_vals) if len(asymmetry_vals) > 0 else None
    
    def _calculate_strides_between_successes(self, df: pd.DataFrame) -> Optional[float]:
        """Calculate average strides between successful trials."""
        if df is None or 'Success' not in df.columns:
            return None
        
        df = df.reset_index(drop=True)
        success_positions = df.index[df['Success'] == 1].tolist()
        
        if len(success_positions) < 2:
            return None
        
        return np.mean(np.diff(success_positions))

class DataManager:
    """Manages all data loading and processing operations."""
    
    def __init__(self, metadata_path: str, data_root_dir: str, config: Config, force_reprocess: bool = False):
        self.metadata_path = metadata_path
        self.data_root_dir = Path(data_root_dir)
        self.config = config
        self.processor = DataProcessor(config)
        self.metrics_calc = MetricsCalculator(config)
        
        self.subjects: Dict[str, Dict] = {}
        self.metadata: pd.DataFrame = None
        
        if not force_reprocess and config.processed_data_file.exists():
            self._load_processed_data()
        else:
            self._process_all_data()
            self._save_processed_data()
    
    def _load_processed_data(self):
        """Load previously processed data."""
        try:
            with open(self.config.processed_data_file, 'rb') as f:
                self.subjects = pickle.load(f)
            
            # Rebuild metadata DataFrame
            self.metadata = pd.DataFrame.from_dict(
                {subj: data['metadata'] for subj, data in self.subjects.items()}, 
                orient='index'
            )
            print(f"✓ Loaded {len(self.subjects)} subjects from cache")
        except Exception as e:
            print(f"Failed to load cached data: {e}")
            self._process_all_data()
            self._save_processed_data()
    
    def _save_processed_data(self):
        """Save processed data."""
        try:
            with open(self.config.processed_data_file, 'wb') as f:
                pickle.dump(self.subjects, f)
            print(f"✓ Saved processed data")
        except Exception as e:
            print(f"Failed to save processed data: {e}")
    
    def _process_all_data(self):
        """Process all subject data."""
        self._load_metadata()
        total_subjects = len(self.metadata)
        
        print(f"Processing {total_subjects} subjects...")
        
        for i, (_, row) in enumerate(self.metadata.iterrows(), 1):
            subject_id = row['ID']
            if i % 10 == 0:
                print(f"Progress: {i}/{total_subjects}")
            
            subject_data = self._process_subject(subject_id, row)
            if subject_data:
                self.subjects[subject_id] = subject_data
    
    def _load_metadata(self):
        """Load and clean metadata."""
        self.metadata = pd.read_csv(self.metadata_path)
        self.metadata['DOB'] = pd.to_datetime(self.metadata['DOB'], errors='coerce')
        self.metadata['Session Date'] = pd.to_datetime(self.metadata['Session Date'], errors='coerce')
        self.metadata = self.metadata.dropna(subset=['ID', 'age_months'])
    
    def _process_subject(self, subject_id: str, metadata_row: pd.Series) -> Optional[Dict]:
        """Process data for a single subject."""
        subject_dir = self.data_root_dir / subject_id
        if not subject_dir.exists():
            return None
        
        trial_data = {}
        
        for original_type, mapped_type in self.config.trial_type_mapping.items():
            df = self.processor.process_trial_files(subject_dir, original_type)
            
            if df is not None:
                if original_type != 'pref':
                    df, anomalies = self._process_trial_data(df)
                else:
                    df = df.drop_duplicates(subset='Left heel strike', keep='last')
                    anomalies = {}
                
                trial_data[mapped_type] = {
                    'data': df,
                    'anomalies': anomalies
                }
        
        if not trial_data:
            return None
        
        return {
            'subject_id': subject_id,
            'metadata': metadata_row.to_dict(),
            'trial_data': trial_data
        }
    
    def _process_trial_data(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        """Process trial data and calculate derived metrics."""
        if df is None or df.empty:
            return None, {}
        
        required_cols = ['Stride Number', 'Success', 'Upper bound success', 'Lower bound success', 'Constant']
        if not all(col in df.columns for col in required_cols):
            return None, {}
        
        df = df.sort_values('Stride Number')
        df['Target size'] = df['Upper bound success'] - df['Lower bound success']
        df = df.drop_duplicates(subset='Stride Number', keep='last')
        
        # Scale sum of gains and steps
        if 'Sum of gains and steps' in df.columns:
            df['Sum of gains and steps'] = 1.5 * df['Sum of gains and steps']
        
        # Detect anomalies
        df, anomalies = self.processor.detect_anomalies(df)
        
        return df, anomalies
    
    def calculate_metrics(self) -> pd.DataFrame:
        """Calculate metrics for all subjects."""
        results = []
        
        print(f"Calculating metrics for {len(self.subjects)} subjects...")
        
        for subject_id, subject in self.subjects.items():
            result = self._calculate_subject_metrics(subject)
            if result:
                results.append(result)
        
        if not results:
            print("No valid metrics calculated!")
            return pd.DataFrame()
        
        df = pd.DataFrame(results).infer_objects()
        print(f"✓ Successfully calculated metrics for {len(df)} subjects")
        return df
    
    def _calculate_subject_metrics(self, subject: Dict) -> Optional[Dict]:
        """Calculate metrics for a single subject."""
        result = {
            'ID': subject['subject_id'],
            'age': subject['metadata'].get('age_months', np.nan) / 12,
            'session_date': subject['metadata'].get('Session Date'),
            'min_const_tape_score': subject['metadata'].get('min_const_tape_score'),
            'pref_const_tape_score': subject['metadata'].get('pref_const_tape_score'), 
            'max_const_tape_score': subject['metadata'].get('max_const_tape_score')
        }
        
        # Process each trial type
        for trial_type in ['vis1', 'invis', 'vis2']:
            trial_dict = subject['trial_data'].get(trial_type)
            if not trial_dict:
                continue
            
            df = trial_dict['data']
            if df is None or df.empty or 'Success' not in df.columns:
                continue
            
            # Calculate standard metrics for both conditions
            for condition in ['max', 'min']:
                period_data, indices = self._get_period_data(df, condition)
                if period_data is not None and not period_data.empty:
                    metrics = self.metrics_calc.calculate_period_metrics(period_data, trial_type, condition)
                    result.update(metrics)
                    result[f'{trial_type}_{condition}_const_indices'] = indices
            
            # Add trial metadata
            if df is not None:
                result.update({
                    f'{trial_type}_min_target_size': df['Target size'].min() if 'Target size' in df.columns else None,
                    f'{trial_type}_max_constant': df['Constant'].max() if 'Constant' in df.columns else None,
                    f'{trial_type}_min_constant': df['Constant'].min() if 'Constant' in df.columns else None
                })
                
                # Order information for invis trials
                if trial_type == 'invis':
                    result.update(self._calculate_condition_order(df))
            
            # Calculate retention metrics for invisible trials
            if trial_type == 'invis':
                retention_metrics = self.metrics_calc.calculate_retention_metrics(trial_dict, trial_type)
                result.update(retention_metrics)
        
        # Process preference trial
        pref_dict = subject['trial_data'].get('pref')
        if pref_dict:
            pref_metrics = self.metrics_calc.calculate_preference_metrics(pref_dict['data'])
            result.update(pref_metrics)
        
        return result
    
    def _get_period_data(self, df: pd.DataFrame, condition: str, length: int = 20) -> Tuple[Optional[pd.DataFrame], Optional[List]]:
        """Extract data for specific condition period."""
        if df is None or df.empty:
            return None, None
        
        if 'Target size' not in df.columns or 'Constant' not in df.columns:
            return None, None
        
        min_target = df['Target size'].min()
        target_tolerance = 0.001
        
        min_target_periods = df[df['Target size'] <= min_target + target_tolerance]
        if min_target_periods.empty:
            return None, None
        
        const_value = (min_target_periods['Constant'].max() if condition == 'max'
                      else min_target_periods['Constant'].min())
        
        period_data = min_target_periods[
            np.isclose(min_target_periods['Constant'], const_value, rtol=1e-5)
        ]
        
        if period_data.empty:
            return None, None
        
        return period_data.tail(length), period_data.index.tolist()
    
    def _calculate_condition_order(self, df: pd.DataFrame) -> Dict:
        """Determine which condition came first for invis trials."""
        all_max_indices = df.index[df['Constant'] == df['Constant'].max()].tolist()
        all_min_indices = df.index[df['Constant'] == df['Constant'].min()].tolist()
        
        if all_max_indices and all_min_indices:
            first_max = min(all_max_indices)
            first_min = min(all_min_indices)
            return {
                'invis_max_first': first_max < first_min,
                'invis_min_first': first_min < first_max
            }
        
        return {'invis_max_first': False, 'invis_min_first': False}
    
    def filter_subjects(self, max_target_size: float = None, min_age: float = None, 
                       max_age: float = None, required_trial_types: List[str] = None) -> 'DataManager':
        """Create a filtered DataManager instance."""
        filtered_subjects = {}
        
        for subject_id, subject in self.subjects.items():
            # Age filtering
            age = subject['metadata'].get('age_months', np.nan) / 12
            if min_age is not None and age < min_age:
                continue
            if max_age is not None and age > max_age:
                continue
            
            # Check required trial types
            if required_trial_types:
                missing_trials = [
                    t for t in required_trial_types 
                    if t not in subject['trial_data'] or subject['trial_data'][t]['data'] is None
                ]
                if missing_trials:
                    continue
            
            # Apply other filters
            valid = True
            for trial_type, trial_dict in subject['trial_data'].items():
                if trial_dict and trial_dict['data'] is not None:
                    df = trial_dict['data']
                    
                    if (max_target_size is not None and 
                        'Target size' in df.columns and 
                        df['Target size'].min() > max_target_size):
                        valid = False
                        break
            
            if valid:
                filtered_subjects[subject_id] = subject
        
        # Create new instance with filtered data
        new_manager = DataManager.__new__(DataManager)
        new_manager.config = self.config
        new_manager.metadata_path = self.metadata_path
        new_manager.data_root_dir = self.data_root_dir
        new_manager.processor = self.processor
        new_manager.metrics_calc = self.metrics_calc
        new_manager.subjects = filtered_subjects
        new_manager.metadata = pd.DataFrame.from_dict(
            {subj: data['metadata'] for subj, data in filtered_subjects.items()}, 
            orient='index'
        )
        
        return new_manager

class StatisticalAnalyzer:
    """Handles all statistical analyses."""
    
    def __init__(self, metrics_df: pd.DataFrame, config: Config):
        self.config = config
        self.metrics_df = metrics_df
        
        # Apply motor noise filter
        if 'mot_noise' in metrics_df.columns:
            self.filtered_df = metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold]
        else:
            self.filtered_df = metrics_df

    def run_repeated_measures_anova(self) -> Dict:
        """Run repeated measures ANOVA with covariates."""
        long_rows = []
        
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            for col in self.filtered_df.columns:
                if '_sr_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type = parts[0]
                        condition = parts[2]
                        
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id,
                                'trial_type': trial_type,
                                'condition': condition,
                                'success_rate': row[col],
                                'age': row.get('age', np.nan),
                                'mot_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan),
                                'min_const_tape_score': row.get('min_const_tape_score', np.nan),
                                'pref_const_tape_score': row.get('pref_const_tape_score', np.nan),
                                'max_const_tape_score': row.get('max_const_tape_score', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna(subset=['success_rate'])
        
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        results = {}
        
        try:
            # Main effects
            if len(df_long['trial_type'].unique()) > 1:
                aov_trial = pg.rm_anova(data=df_long, dv='success_rate', 
                                       within='trial_type', subject='subject', detailed=True)
                results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size': float(aov_trial['ng2'].iloc[0])
                }
            
            if len(df_long['condition'].unique()) > 1:
                aov_condition = pg.rm_anova(data=df_long, dv='success_rate', 
                                           within='condition', subject='subject', detailed=True)
                results['condition_effect'] = {
                    'F': float(aov_condition['F'].iloc[0]),
                    'p_value': float(aov_condition['p-unc'].iloc[0]),
                    'effect_size': float(aov_condition['ng2'].iloc[0])
                }
            
            # Interaction
            if len(df_long['trial_type'].unique()) > 1 and len(df_long['condition'].unique()) > 1:
                aov_interaction = pg.rm_anova(data=df_long, dv='success_rate', 
                                             within=['trial_type', 'condition'], 
                                             subject='subject', detailed=True)
                interaction_row = aov_interaction[aov_interaction['Source'].str.contains('trial_type \\* condition')]
                if not interaction_row.empty:
                    results['interaction_effect'] = {
                        'F': float(interaction_row['F'].iloc[0]),
                        'p_value': float(interaction_row['p-unc'].iloc[0]),
                        'effect_size': float(interaction_row['ng2'].iloc[0])
                    }
        except Exception as e:
            results['basic_anova_error'] = str(e)
        
        # Covariate analysis
        try:
            available_covariates = []
            for cov in ['age', 'mot_noise', 'pref_asymmetry', 'min_const_tape_score', 
                       'pref_const_tape_score', 'max_const_tape_score']:
                if cov in df_long.columns and df_long[cov].notna().sum() > 0:
                    available_covariates.append(cov)
            
            if available_covariates:
                results['ancova_covariates'] = available_covariates
                
                # Covariate correlations
                for cov in available_covariates:
                    cov_corr = df_long.groupby('subject').agg({
                        'success_rate': 'mean',
                        cov: 'first'
                    }).reset_index()
                    if len(cov_corr) > 5:
                        r, p = pearsonr(cov_corr[cov], cov_corr['success_rate'])
                        results[f'{cov}_covariate_effect'] = {
                            'correlation': float(r),
                            'p_value': float(p),
                            'interpretation': f'{cov} effect on success rate'
                        }
                
                # Mixed effects model
                try:
                    covariate_terms = ' + '.join(available_covariates)
                    formula = f"success_rate ~ trial_type * condition + {covariate_terms}"
                    
                    model = smf.mixedlm(formula, df_long, groups=df_long['subject'])
                    fitted_model = model.fit()
                    
                    covariate_effects = {}
                    for cov in available_covariates:
                        if cov in fitted_model.params.index:
                            covariate_effects[f'{cov}_effect'] = {
                                'coefficient': float(fitted_model.params[cov]),
                                'p_value': float(fitted_model.pvalues[cov]),
                                'confidence_interval': [float(fitted_model.conf_int().loc[cov, 0]), 
                                                      float(fitted_model.conf_int().loc[cov, 1])],
                                'interpretation': f'Change in success rate per unit increase in {cov}'
                            }
                    
                    results['mixed_effects_covariates'] = covariate_effects
                    
                except Exception as e:
                    results['mixed_effects_error'] = str(e)
            
        except Exception as e:
            results['covariate_analysis_error'] = str(e)
        
        return results

    def run_age_stratified_anova(self, age_groups: Dict[str, List[float]] = None, 
                                  min_subjects_per_group: int = 1) -> Dict:
        """Run ANOVA analysis stratified by age groups."""
        
        if age_groups is None:
            age_groups = {
                'younger': [7, 12],
                'middle': [12, 15], 
                'older': [15, 18]
            }
        
        # Prepare long-format data
        long_rows = []
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            age = row.get('age', np.nan)
            
            if pd.isna(age):
                continue
                
            for col in self.filtered_df.columns:
                if '_sr_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type = parts[0]
                        condition = parts[2]
                        
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id,
                                'trial_type': trial_type,
                                'condition': condition,
                                'success_rate': row[col],
                                'age': age,
                                'mot_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna(subset=['age', 'success_rate'])
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        results = {
            'age_group_definitions': age_groups,
            'min_subjects_threshold': min_subjects_per_group,
            'total_subjects_analyzed': len(df_long['subject'].unique()),
            'age_range': [float(df_long['age'].min()), float(df_long['age'].max())],
            'group_analyses': {}
        }
        
        # Assign subjects to age groups
        df_long['age_group'] = None
        group_assignments = {}
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df_long['age'] >= min_age) & (df_long['age'] < max_age)
            df_long.loc[mask, 'age_group'] = group_name
            
            subjects_in_group = df_long[mask]['subject'].unique()
            group_assignments[group_name] = {
                'subjects': list(subjects_in_group),
                'n_subjects': len(subjects_in_group),
                'age_range': [float(df_long[mask]['age'].min()) if len(subjects_in_group) > 0 else np.nan,
                             float(df_long[mask]['age'].max()) if len(subjects_in_group) > 0 else np.nan],
                'mean_age': float(df_long[mask]['age'].mean()) if len(subjects_in_group) > 0 else np.nan
            }
        
        results['group_assignments'] = group_assignments
        
        # Run ANOVA for each age group
        for group_name, group_info in group_assignments.items():
            if group_info['n_subjects'] < min_subjects_per_group:
                results['group_analyses'][group_name] = {
                    'error': f'Insufficient subjects (n={group_info["n_subjects"]}, minimum={min_subjects_per_group})'
                }
                continue
            
            group_data = df_long[df_long['age_group'] == group_name].copy()
            group_results = {}
            
            try:
                aov_trial = pg.rm_anova(data=group_data, dv='success_rate', 
                                       within='trial_type', subject='subject', detailed=True)
                group_results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size_eta2': float(aov_trial['ng2'].iloc[0]),
                    'significant': float(aov_trial['p-unc'].iloc[0]) < 0.05
                }
                
            except Exception as e:
                group_results['anova_error'] = str(e)
            
            # Post-hoc pairwise comparisons
            try:
                if len(group_data['trial_type'].unique()) > 2:
                    subject_means = group_data.groupby(['subject', 'trial_type'])['success_rate'].mean().reset_index()
                    posthoc = pg.pairwise_ttests(data=subject_means, dv='success_rate', 
                                               within='trial_type', subject='subject',
                                               padjust='bonf')
                    
                    group_results['posthoc_comparisons'] = {}
                    for _, row in posthoc.iterrows():
                        comparison = f"{row['A']}_vs_{row['B']}"
                        group_results['posthoc_comparisons'][comparison] = {
                            'mean_diff': float(row.get('mean(A)', 0) - row.get('mean(B)', 0)) if 'mean(A)' in row else 0.0,
                            't_stat': float(row['T']),
                            'p_corrected': float(row['p-corr']),
                            'cohens_d': float(row.get('hedges', 0.0)),
                            'significant': row['p-corr'] < 0.05,
                            'effect_size_interpretation': self._interpret_cohens_d(float(row['hedges']))
                        }
            except Exception as e:
                group_results['posthoc_error'] = str(e)
            
            # Descriptive statistics
            try:
                group_results['mean_success_rates'] = {
                    trial: float(group_data[group_data['trial_type'] == trial]['success_rate'].mean())
                    for trial in group_data['trial_type'].unique()
                }
                
                group_results['descriptive_stats'] = {
                    'n_subjects': group_info['n_subjects'],
                    'n_observations': len(group_data),
                    'age_info': {
                        'mean_age': group_info['mean_age'],
                        'age_range': group_info['age_range']
                    }
                }
                
            except Exception as e:
                group_results['descriptive_error'] = str(e)
            
            results['group_analyses'][group_name] = group_results
        
        return results

    def run_regression_analysis(self, trial_type: str = 'invis', condition: str = 'max',
                               predictors: List[str] = None) -> Dict:
        """Run regression analysis."""
        if predictors is None:
            predictors = ['age', 'mot_noise', 'pref_asymmetry']
        
        available_predictors = [p for p in predictors if p in self.filtered_df.columns]
        target_col = f'{trial_type}_sr_{condition}_const'
        
        if target_col not in self.filtered_df.columns:
            raise ValueError(f"Target column {target_col} not found")
        
        # Prepare data
        valid_data = self.filtered_df[available_predictors + [target_col]].dropna()
        
        if len(valid_data) < 10:
            raise ValueError(f"Insufficient data: only {len(valid_data)} valid samples")
        
        X = valid_data[available_predictors]
        y = valid_data[target_col]
        
        # Split and train
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', LinearRegression())
        ])
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        return {
            'model': model,
            'metrics': {
                'r2': r2_score(y_test, y_pred),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                'n_samples': len(valid_data)
            },
            'feature_importances': dict(zip(available_predictors, 
                                          np.abs(model.named_steps['regressor'].coef_)))
        }

    def run_correlation_analysis(self) -> Dict:
        """Run correlation analysis between key variables."""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        # Add tape scores
        tape_scores = ['min_const_tape_score', 'pref_const_tape_score', 'max_const_tape_score']
        for tape_score in tape_scores:
            if tape_score in self.filtered_df.columns:
                key_cols.append(tape_score)
        
        # Add success rate columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols)
        
        # Calculate correlation matrix
        corr_data = self.filtered_df[key_cols].corr()
        
        # Extract significant correlations
        significant_corrs = {}
        for i, col1 in enumerate(key_cols):
            for j, col2 in enumerate(key_cols):
                if i < j:  # Avoid duplicates
                    valid_data = self.filtered_df[[col1, col2]].dropna()
                    if len(valid_data) > 5:
                        r, p = pearsonr(valid_data[col1], valid_data[col2])
                        if p < 0.05:
                            significant_corrs[f'{col1}_vs_{col2}'] = {
                                'r': r,
                                'p': p,
                                'n': len(valid_data)
                            }
        
        return {
            'correlation_matrix': corr_data.to_dict(),
            'significant_correlations': significant_corrs
        }

    def _interpret_cohens_d(self, d):
        """Interpret Cohen's d effect size."""
        abs_d = abs(d)
        if abs_d < 0.2:
            return "negligible"
        elif abs_d < 0.5:
            return "small"
        elif abs_d < 0.8:
            return "medium"
        else:
            return "large"

class Plotter:
    """Unified plotting class for all visualizations."""
    
    def __init__(self, metrics_df: pd.DataFrame, config: Config, data_manager: DataManager = None):
        self.metrics_df = metrics_df
        self.config = config
        self.data_manager = data_manager
        
        # Apply motor noise filter
        if 'mot_noise' in metrics_df.columns:
            self.filtered_df = metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold]
        else:
            self.filtered_df = metrics_df
        
        self.colors = {
            'primary': '#667eea', 'secondary': '#764ba2', 'success': '#28a745',
            'warning': '#ffc107', 'danger': '#dc3545', 'vis1': '#1f77b4', 
            'invis': '#ff7f0e', 'vis2': '#2ca02c'
        }

    def save_figure(self, fig: plt.Figure, filename: str):
        """Save figure to appropriate directory."""
        save_path = self.config.figures_dir / filename
        fig.savefig(save_path, dpi=self.config.figure_dpi, bbox_inches='tight')
        plt.close(fig)
        return save_path

    def add_trendline(self, ax, x, y):
        """Add trendline with correlation to plot."""
        x_clean = pd.to_numeric(x, errors='coerce')
        y_clean = pd.to_numeric(y, errors='coerce')
        valid = x_clean.notna() & y_clean.notna()
        
        if valid.sum() < 2:
            return
        
        x_vals = x_clean[valid]
        y_vals = y_clean[valid]
        
        try:
            coeffs = np.polyfit(x_vals, y_vals, 1)
            trendline = np.poly1d(coeffs)
            r, p = pearsonr(x_vals, y_vals)
            
            ax.plot(x_vals, trendline(x_vals), 'r--', alpha=0.8, linewidth=2)
            ax.text(0.05, 0.95, f'r² = {r**2:.3f}\np = {p:.3f}\nn = {len(x_vals)}', 
                   transform=ax.transAxes,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                   verticalalignment='top', fontsize=10)
        except Exception:
            pass

    def plot_age_vs_success_rates(self) -> Path:
        """Plot age vs success rates by trial/condition with motor noise coloring."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_sr_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel('Success Rate')
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.set_ylim(-0.05, 1.05)
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
        
        plt.suptitle('Age vs Success Rates by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'age_vs_success_rates.png')

    def plot_correlation_matrix(self) -> Path:
        """Plot correlation matrix of key variables."""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        # Add success rate columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols[:6])  # Limit to first 6 to avoid clutter
        
        if len(key_cols) > 1:
            fig = plt.figure(figsize=(12, 10))
            
            corr_matrix = self.filtered_df[key_cols].corr()
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            
            sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                       square=True, linewidths=0.5, fmt='.2f')
            
            plt.title('Correlation Matrix of Key Variables', fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            return self.save_figure(fig, 'correlation_matrix.png')

    def plot_age_stratified_results(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """Create comprehensive visualization of age-stratified ANOVA results."""
        
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'trial_type_effect' in results]
        
        if not valid_groups:
            print("No valid age groups found for visualization")
            return None
        
        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(3, max(len(valid_groups), 3), hspace=0.4, wspace=0.3)
        
        trial_colors = {'vis1': '#1f77b4', 'invis': '#ff7f0e', 'vis2': '#2ca02c'}
        
        # Row 1: Mean success rates by age group and trial type
        for i, group in enumerate(valid_groups):
            ax = fig.add_subplot(gs[0, i])
            group_data = age_results['group_analyses'][group]
            
            if 'mean_success_rates' in group_data:
                trials = list(group_data['mean_success_rates'].keys())
                means = list(group_data['mean_success_rates'].values())
                colors = [trial_colors.get(trial, 'gray') for trial in trials]
                
                bars = ax.bar(trials, means, color=colors, alpha=0.7, edgecolor='black', linewidth=1)
                
                for bar, mean in zip(bars, means):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                           f'{mean:.2f}', ha='center', va='bottom', fontweight='bold')
                
                ax.set_ylim(0, 1)
                ax.set_ylabel('Mean Success Rate')
                ax.set_title(f'{group.title()}\n(n={group_data.get("descriptive_stats", {}).get("n_subjects", "?")})')
                ax.grid(True, alpha=0.3)
        
        # Row 2: Effect sizes comparison
        ax_effect = fig.add_subplot(gs[1, :len(valid_groups)])
        
        effect_sizes = []
        group_names = []
        significance = []
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                effect_sizes.append(group_data['trial_type_effect']['effect_size_eta2'])
                group_names.append(group.title())
                significance.append(group_data['trial_type_effect']['significant'])
        
        if effect_sizes:
            colors = ['green' if sig else 'red' for sig in significance]
            bars = ax_effect.bar(group_names, effect_sizes, color=colors, alpha=0.7, edgecolor='black')
            
            for i, (bar, sig, eta2) in enumerate(zip(bars, significance, effect_sizes)):
                height = bar.get_height()
                sig_text = '***' if sig else 'ns'
                ax_effect.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                              f'{eta2:.3f}\n{sig_text}', ha='center', va='bottom', fontweight='bold')
            
            ax_effect.set_ylabel('Effect Size (η²)')
            ax_effect.set_title('Trial Type Effect Sizes by Age Group')
            ax_effect.grid(True, alpha=0.3)
            
            # Add effect size interpretation lines
            ax_effect.axhline(y=0.01, color='gray', linestyle='--', alpha=0.5, label='Small (0.01)')
            ax_effect.axhline(y=0.06, color='orange', linestyle='--', alpha=0.5, label='Medium (0.06)')
            ax_effect.axhline(y=0.14, color='red', linestyle='--', alpha=0.5, label='Large (0.14)')
            ax_effect.legend(loc='upper right')
        
        # Row 3: Summary table
        ax_table = fig.add_subplot(gs[2, :])
        ax_table.axis('off')
        
        table_data = []
        headers = ['Age Group', 'N', 'Mean Age', 'F-stat', 'p-value', 'η²', 'Significant']
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data and 'descriptive_stats' in group_data:
                trial_effect = group_data['trial_type_effect']
                desc_stats = group_data['descriptive_stats']
                
                row = [
                    group.title(),
                    str(desc_stats['n_subjects']),
                    f"{desc_stats['age_info']['mean_age']:.1f}y",
                    f"{trial_effect['F']:.2f}",
                    f"{trial_effect['p_value']:.3f}" if trial_effect['p_value'] >= 0.001 else "<0.001",
                    f"{trial_effect['effect_size_eta2']:.3f}",
                    "Yes" if trial_effect['significant'] else "No"
                ]
                table_data.append(row)
        
        if table_data:
            table = ax_table.table(cellText=table_data,
                                  colLabels=headers,
                                  cellLoc='center',
                                  loc='center',
                                  bbox=[0, 0.3, 1, 0.6])
            
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            table.scale(1, 1.5)
            
            # Color code significance
            for i in range(len(table_data)):
                if table_data[i][6] == "Yes":  # Significant
                    table[(i+1, 6)].set_facecolor('#90EE90')  # Light green
                else:
                    table[(i+1, 6)].set_facecolor('#FFB6C1')  # Light red
            
            # Style header
            for j in range(len(headers)):
                header_cell = table[(0, j)]
                header_cell.set_facecolor('#4CAF50')
                header_cell.set_text_props(weight='bold', color='white')
        
        plt.suptitle('Age-Stratified ANOVA Results: Trial Type Effects', fontsize=16, fontweight='bold')
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Figure saved to: {save_path}")
        
        return fig

    def plot_learning_distributions(self) -> Path:
        """Plot learning metric distributions by trial and condition."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Plot by trial type (combining conditions)
        for i, trial in enumerate(trials):
            ax = axes[0, i]
            learning_data = []
            labels = []
            colors = []
            
            for condition in conditions:
                col = f'{trial}_learning_{condition}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    if not data.empty:
                        learning_data.append(data)
                        labels.append(f'{condition.capitalize()}')
                        colors.append('lightblue' if condition == 'max' else 'lightcoral')
            
            if learning_data:
                bp = ax.boxplot(learning_data, labels=labels, patch_artist=True)
                for patch, color in zip(bp['boxes'], colors):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                
                # Add reference lines
                ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No Learning')
                ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect Learning')
            
            ax.set_title(f'{trial.upper()} Trial')
            ax.set_ylabel('Learning')
            ax.grid(True, alpha=0.3)
            if i == 0:
                ax.legend()
        
        # Plot by condition (combining trials)  
        for j, condition in enumerate(conditions):
            ax = axes[1, j]
            learning_data = []
            labels = []
            colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, orange, green
            
            for trial in trials:
                col = f'{trial}_learning_{condition}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    if not data.empty:
                        learning_data.append(data)
                        labels.append(trial.upper())
            
            if learning_data:
                bp = ax.boxplot(learning_data, labels=labels, patch_artist=True)
                for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                
                # Add reference lines
                ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No Learning')
                ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect Learning')
            
            ax.set_title(f'{condition.capitalize()} Target Condition')
            ax.set_ylabel('Learning')
            ax.grid(True, alpha=0.3)
            if j == 0:
                ax.legend()
        
        # Overall distribution in bottom right
        ax = axes[1, 2]
        all_learning_data = []
        for trial in trials:
            for condition in conditions:
                col = f'{trial}_learning_{condition}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    all_learning_data.extend(data.values)
        
        if all_learning_data:
            ax.hist(all_learning_data, bins=30, alpha=0.7, edgecolor='black')
            ax.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='No Learning')
            ax.axvline(x=1, color='green', linestyle='--', alpha=0.7, label='Perfect Learning')
            ax.axvline(x=np.mean(all_learning_data), color='blue', linestyle='-', alpha=0.7, 
                      label=f'Mean = {np.mean(all_learning_data):.3f}')
        
        ax.set_title('Overall Learning Distribution')
        ax.set_xlabel('Learning')
        ax.set_ylabel('Frequency')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.suptitle('Learning Metric Distributions\nLearning = 0: No adaptation, Learning = 1: Perfect adaptation to target', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'learning_distributions.png')

    def plot_retention_analysis(self) -> List[Path]:
        """Plot retention analysis for invisible trial type."""
        plot_paths = []
        
        # Check if we have retention data
        retention_cols = [col for col in self.filtered_df.columns if 'retention' in col]
        if not retention_cols:
            print("No retention data found")
            return plot_paths
        
        try:
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            
            # Box plot comparing clamp1 vs clamp2
            ax = axes[0]
            retention_data = []
            labels = []
            
            for clamp in ['clamp1', 'clamp2']:
                col = f'invis_retention_{clamp}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    if not data.empty:
                        retention_data.append(data)
                        labels.append(clamp.capitalize())
            
            if retention_data:
                bp = ax.boxplot(retention_data, labels=labels, patch_artist=True)
                colors = ['lightblue', 'lightcoral']
                for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                
                ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No Retention')
                ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect Retention')
                ax.set_title('Retention by Clamp Period')
                ax.set_ylabel('Retention')
                ax.legend()
                ax.grid(True, alpha=0.3)
            
            # Retention vs age
            ax = axes[1]
            for clamp, color in [('clamp1', 'blue'), ('clamp2', 'red')]:
                col = f'invis_retention_{clamp}_const'
                if col in self.filtered_df.columns:
                    valid_data = self.filtered_df[['age', col]].dropna()
                    if not valid_data.empty:
                        ax.scatter(valid_data['age'], valid_data[col], 
                                 alpha=0.6, s=60, color=color, label=clamp.capitalize())
            
            ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
            ax.axhline(y=1, color='green', linestyle='--', alpha=0.5)
            ax.set_xlabel('Age (years)')
            ax.set_ylabel('Retention')
            ax.set_title('Retention vs Age')
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            # Overall retention histogram
            ax = axes[2]
            if 'invis_retention_overall_const' in self.filtered_df.columns:
                data = self.filtered_df['invis_retention_overall_const'].dropna()
                if not data.empty:
                    ax.hist(data, bins=20, alpha=0.7, edgecolor='black')
                    ax.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='No Retention')
                    ax.axvline(x=1, color='green', linestyle='--', alpha=0.7, label='Perfect Retention')
                    ax.axvline(x=data.mean(), color='blue', linestyle='-', alpha=0.7, 
                              label=f'Mean = {data.mean():.3f}')
            
            ax.set_xlabel('Overall Retention')
            ax.set_ylabel('Frequency')
            ax.set_title('Overall Retention Distribution')
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            plt.suptitle('Retention Analysis: Success Clamp Performance\n(Retention = 0: No retention, Retention = 1: Perfect retention)', 
                         fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            plot_path = self.config.figures_dir / 'retention_analysis.png'
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close()
            plot_paths.append(plot_path)
            
        except Exception as e:
            print(f"Failed to create retention analysis plot: {e}")
        
        return plot_paths

    def plot_subject_summary(self, subject_id: str) -> Optional[Path]:
        """Plot comprehensive summary for a subject."""
        if not self.data_manager or subject_id not in self.data_manager.subjects:
            return None
        
        subject = self.data_manager.subjects[subject_id]
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Success rates across trials
        ax1 = axes[0, 0]
        trial_types = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        
        x_pos = np.arange(len(trial_types))
        width = 0.35
        
        max_rates = []
        min_rates = []
        
        for trial in trial_types:
            trial_dict = subject['trial_data'].get(trial)
            if trial_dict and trial_dict['data'] is not None:
                df = trial_dict['data']
                
                # Get success rates for both conditions
                max_data, _ = self.data_manager._get_period_data(df, 'max')
                min_data, _ = self.data_manager._get_period_data(df, 'min')
                
                max_rate = max_data['Success'].mean() if max_data is not None else 0
                min_rate = min_data['Success'].mean() if min_data is not None else 0
                
                max_rates.append(max_rate)
                min_rates.append(min_rate)
            else:
                max_rates.append(0)
                min_rates.append(0)
        
        ax1.bar(x_pos - width/2, max_rates, width, label='Max Target', alpha=0.8)
        ax1.bar(x_pos + width/2, min_rates, width, label='Min Target', alpha=0.8)
        ax1.set_xlabel('Trial Type')
        ax1.set_ylabel('Success Rate')
        ax1.set_title('Success Rates by Trial Type')
        ax1.set_xticks(x_pos)
        ax1.set_xticklabels([t.upper() for t in trial_types])
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Motor noise visualization (if available)
        ax2 = axes[0, 1]
        pref_dict = subject['trial_data'].get('pref')
        if pref_dict and pref_dict['data'] is not None:
            pref_df = pref_dict['data']
            if all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
                right_steps = pref_df['Right step length']
                left_steps = pref_df['Left step length']
                
                ax2.plot(right_steps, label='Right Steps', alpha=0.7)
                ax2.plot(left_steps, label='Left Steps', alpha=0.7)
                ax2.set_xlabel('Stride Number')
                ax2.set_ylabel('Step Length')
                ax2.set_title('Preference Trial Step Lengths')
                ax2.legend()
                ax2.grid(True, alpha=0.3)
        else:
            ax2.text(0.5, 0.5, 'No Preference Data', ha='center', va='center',
                    transform=ax2.transAxes, fontsize=12)
        
        # Plot 3: Age comparison
        ax3 = axes[1, 0]
        all_ages = [self.data_manager.subjects[s]['metadata'].get('age_months', np.nan) / 12 
                   for s in self.data_manager.subjects.keys()]
        all_ages = [age for age in all_ages if not pd.isna(age)]
        
        ax3.hist(all_ages, bins=20, alpha=0.7, label='All Subjects')
        subject_age = subject['metadata'].get('age_months', np.nan) / 12
        ax3.axvline(subject_age, color='red', linestyle='--', linewidth=2, 
                   label=f'This Subject (Age: {subject_age:.1f})')
        ax3.set_xlabel('Age (years)')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Age Distribution')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Subject information
        ax4 = axes[1, 1]
        ax4.text(0.5, 0.5, f'Subject ID: {subject_id}\nAge: {subject_age:.1f} years\n'
                f'Session: {subject["metadata"].get("Session Date", "Unknown")}', 
                ha='center', va='center', transform=ax4.transAxes, fontsize=12,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        ax4.set_title('Subject Information')
        ax4.axis('off')
        
        plt.suptitle(f'Subject {subject_id} Summary', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        filename = f'subject_summary_{subject_id}.png'
        return self.save_figure(fig, filename)

class MotorLearningAnalysis:
    """Main analysis interface that coordinates all components."""
    
    def __init__(self, config: Config = None):
        self.config = config or Config()
        self.data_manager: Optional[DataManager] = None
        self.metrics_df: Optional[pd.DataFrame] = None
        self.statistical_analyzer: Optional[StatisticalAnalyzer] = None
        self.plotter: Optional[Plotter] = None
        self.results: Optional[Dict] = None
    
    def load_data(self, metadata_path: str, data_root_dir: str, 
                  force_reprocess: bool = False) -> 'MotorLearningAnalysis':
        """Load and process data."""
        self.data_manager = DataManager(metadata_path, data_root_dir, self.config, force_reprocess)
        return self
    
    def filter_data(self, **kwargs) -> 'MotorLearningAnalysis':
        """Filter data based on criteria."""
        if self.data_manager:
            self.data_manager = self.data_manager.filter_subjects(**kwargs)
        return self
    
    def calculate_metrics(self) -> 'MotorLearningAnalysis':
        """Calculate metrics for all subjects."""
        if not self.data_manager:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        self.metrics_df = self.data_manager.calculate_metrics()
        
        # Initialize analyzers with metrics
        self.statistical_analyzer = StatisticalAnalyzer(self.metrics_df, self.config)
        self.plotter = Plotter(self.metrics_df, self.config, self.data_manager)
        
        return self
    
    def run_analysis(self, include_visualizations: bool = True) -> 'MotorLearningAnalysis':
        """Run comprehensive analysis and store results."""
        if self.metrics_df is None:
            raise ValueError("Metrics not calculated. Call calculate_metrics() first.")
        
        self.results = {
            'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
            'n_subjects': len(self.metrics_df),
            'analyses': {}
        }
        
        # Statistical analyses
        if self.statistical_analyzer:
            try:
                self.results['analyses']['regression'] = self.statistical_analyzer.run_regression_analysis()
            except Exception as e:
                self.results['analyses']['regression'] = {'error': str(e)}
            
            try:
                self.results['analyses']['anova'] = self.statistical_analyzer.run_repeated_measures_anova()
            except Exception as e:
                self.results['analyses']['anova'] = {'error': str(e)}
            
            try:
                self.results['analyses']['correlations'] = self.statistical_analyzer.run_correlation_analysis()
            except Exception as e:
                self.results['analyses']['correlations'] = {'error': str(e)}
        
        # Visualizations
        if include_visualizations and self.plotter:
            self.results['visualizations'] = []
            
            try:
                plot_path = self.plotter.plot_age_vs_success_rates()
                if plot_path:
                    self.results['visualizations'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create age vs success rates plot: {e}")
            
            try:
                plot_path = self.plotter.plot_correlation_matrix()
                if plot_path:
                    self.results['visualizations'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create correlation matrix: {e}")
            
            try:
                plot_paths = self.plotter.plot_retention_analysis()
                for plot_path in plot_paths:
                    self.results['visualizations'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create retention plots: {e}")
            
            try:
                plot_path = self.plotter.plot_learning_distributions()
                if plot_path:
                    self.results['visualizations'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create learning distributions: {e}")
        
        # Generate report
        self._generate_report()
        
        return self
    
    def get_results(self) -> Dict:
        """Get analysis results."""
        return self.results
    
    def _generate_report(self):
        """Generate analysis report."""
        if not self.results:
            return
        
        report_path = self.config.reports_dir / f"analysis_report_{self.results['timestamp']}.json"
        
        def make_serializable(obj):
            if isinstance(obj, (np.integer, np.floating)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, Path):
                return str(obj)
            elif pd.isna(obj):
                return None
            elif hasattr(obj, '__dict__') and not isinstance(obj, (dict, list, tuple)):
                return str(type(obj).__name__)
            return obj
        
        serializable_results = json.loads(
            json.dumps(self.results, default=make_serializable)
        )
        
        with open(report_path, 'w') as f:
            json.dump(serializable_results, f, indent=2)
        
        print(f"Report saved to: {report_path}")

# Convenience function
def run_motor_learning_analysis(metadata_path: str, data_root_dir: str,
                               output_dir: str = 'motor_learning_output',
                               required_trials: List[str] = None) -> MotorLearningAnalysis:
    """Convenience function to run complete analysis."""
    
    config = Config(base_output_dir=Path(output_dir))
    
    analysis = (MotorLearningAnalysis(config)
                .load_data(metadata_path, data_root_dir)
                .filter_data(required_trial_types=required_trials or ['vis1', 'invis', 'vis2'])
                .calculate_metrics()
                .run_analysis())
    
    return analysis

# Example usage
def main():
    """Example usage of the motor learning analysis pipeline."""
    
    data_root_dir = 'muh_data/'
    metadata_path = 'muh_metadata.csv'
    
    print("Motor Learning Analysis Pipeline - Condensed Version")
    print("=" * 60)
    
    # Run analysis
    analysis = run_motor_learning_analysis(metadata_path, data_root_dir)
    
    # Access components
    data_manager = analysis.data_manager
    metrics_df = analysis.metrics_df
    stats = analysis.statistical_analyzer
    plotter = analysis.plotter
    
    # Run specific analyses
    print("\nRunning specific analyses...")
    regression_results = stats.run_regression_analysis()
    anova_results = stats.run_repeated_measures_anova()
    correlation_results = stats.run_correlation_analysis()
    
    # Run age-stratified analysis
    age_results = stats.run_age_stratified_anova()
    age_plot = plotter.plot_age_stratified_results(age_results, 'age_stratified_results.png')
    
    # Print results
    print("\nRegression Results:")
    print(f"  R²: {regression_results['metrics']['r2']:.3f}")
    print(f"  RMSE: {regression_results['metrics']['rmse']:.3f}")
    print(f"  Sample size: {regression_results['metrics']['n_samples']}")
    
    print("\nANOVA Results:")
    for effect, result in anova_results.items():
        if isinstance(result, dict) and 'F' in result:
            print(f"  {effect}: F={result['F']:.3f}, p={result['p_value']:.3f}")
    
    print(f"\nDataset Summary:")
    print(f"  Total subjects: {len(metrics_df)}")
    print(f"  Age range: {metrics_df['age'].min():.1f} - {metrics_df['age'].max():.1f} years")
    
    # Generate individual plots for first few subjects
    print("\nGenerating individual plots...")
    subject_ids = list(data_manager.subjects.keys())[:3]
    for subject_id in subject_ids:
        try:
            plot_path = plotter.plot_subject_summary(subject_id)
            if plot_path:
                print(f"  Created summary plot for {subject_id}")
        except Exception as e:
            print(f"  Failed to create plot for {subject_id}: {e}")
    
    print(f"\nAnalysis complete!")
    print(f"  All outputs saved to: {analysis.config.base_output_dir}")
    
    return analysis

if __name__ == "__main__":
    # Example usage
    print("Motor Learning Analysis Pipeline - Condensed Version")
    print("=" * 60)
    print("\nThis pipeline provides a comprehensive analysis of motor learning data.")
    print("\nKey features:")
    print("- Automated data loading and preprocessing")
    print("- Comprehensive statistical analysis (regression, ANOVA, correlations)")
    print("- Population and individual visualizations")
    print("- Learning and retention metrics")
    print("- Age-stratified analysis")
    print("\nTo run analysis:")
    print("  analysis = main()")
    print("  # or")
    print("  analysis = run_motor_learning_analysis('metadata.csv', 'data_dir/')")
    
    # Uncomment to run the example
analysis = main()

Motor Learning Analysis Pipeline - Condensed Version

This pipeline provides a comprehensive analysis of motor learning data.

Key features:
- Automated data loading and preprocessing
- Comprehensive statistical analysis (regression, ANOVA, correlations)
- Population and individual visualizations
- Learning and retention metrics
- Age-stratified analysis

To run analysis:
  analysis = main()
  # or
  analysis = run_motor_learning_analysis('metadata.csv', 'data_dir/')
Motor Learning Analysis Pipeline - Condensed Version
✓ Loaded 110 subjects from cache
Calculating metrics for 66 subjects...


KeyError: 'subject_id'